In [1]:
#! pip install --upgrade xarray zarr gcsfs cftime nc-time-axis

In [2]:
#! pip install xarray zarr gcsfs

In [3]:
#pip install dask

In [1]:
import numpy as np
import pandas as pd
import xarray as xr
import zarr
import gcsfs
import matplotlib.pyplot as plt
%matplotlib inline


xr.set_options(display_style='html')
%config InlineBackend.figure_format = 'retina' 

In [2]:
plt.rcParams['figure.figsize'] = 12, 6

In [3]:
df = pd.read_csv('https://storage.googleapis.com/cmip6/cmip6-zarr-consolidated-stores.csv')
df.head()

,activity_id,institution_id,source_id,experiment_id,member_id,table_id,variable_id,grid_label,zstore,dcpp_init_year,version
0,HighResMIP,CMCC,CMCC-CM2-HR4,highresSST-present,r1i1p1f1,Amon,ps,gn,gs://cmip6/CMIP6/HighResMIP/CMCC/CMCC-CM2-HR4/...,NaN,20170706
1,HighResMIP,CMCC,CMCC-CM2-HR4,highresSST-present,r1i1p1f1,Amon,rsds,gn,gs://cmip6/CMIP6/HighResMIP/CMCC/CMCC-CM2-HR4/...,NaN,20170706
2,HighResMIP,CMCC,CMCC-CM2-HR4,highresSST-present,r1i1p1f1,Amon,rlus,gn,gs://cmip6/CMIP6/HighResMIP/CMCC/CMCC-CM2-HR4/...,NaN,20170706
3,HighResMIP,CMCC,CMCC-CM2-HR4,highresSST-present,r1i1p1f1,Amon,rlds,gn,gs://cmip6/CMIP6/HighResMIP/CMCC/CMCC-CM2-HR4/...,NaN,20170706
4,HighResMIP,CMCC,CMCC-CM2-HR4,highresSST-present,r1i1p1f1,Amon,psl,gn,gs://cmip6/CMIP6/HighResMIP/CMCC/CMCC-CM2-HR4/...,NaN,20170706


**

In [19]:
historical_precipitation = df[
   (df['institution_id'] == 'NCAR') &
   (df['source_id'] == 'CESM2') &
   (df['member_id'] == 'r11i1p1f1') &
   (df['activity_id'] == 'CMIP') &
   (df['experiment_id'] == 'historical') &
   (df['variable_id'].isin(['pr'])) &
   (df['table_id'].isin(['Amon'])) 
]
historical_precipitation

,activity_id,institution_id,source_id,experiment_id,member_id,table_id,variable_id,grid_label,zstore,dcpp_init_year,version
200657,CMIP,NCAR,CESM2,historical,r11i1p1f1,Amon,pr,gn,gs://cmip6/CMIP6/CMIP/NCAR/CESM2/historical/r1...,NaN,20190514


In [20]:
# Load historical precipitation data
gcs = gcsfs.GCSFileSystem(token='anon')

# Get the path
zstore = historical_precipitation.zstore.values[0]

# Open the dataset
ds_historical = xr.open_zarr(gcs.get_mapper(zstore), consolidated=True)
ds_historical

<xarray.Dataset> Size: 438MB
Dimensions:    (time: 1980, lat: 192, lon: 288, nbnd: 2)
Coordinates:
  * time       (time) object 16kB 1850-01-15 12:00:00 ... 2014-12-15 12:00:00
  * lat        (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon        (lon) float64 2kB 0.0 1.25 2.5 3.75 ... 355.0 356.2 357.5 358.8
    lat_bnds   (lat, nbnd) float64 3kB dask.array<chunksize=(192, 2), meta=np.ndarray>
    lon_bnds   (lon, nbnd) float64 5kB dask.array<chunksize=(288, 2), meta=np.ndarray>
    time_bnds  (time, nbnd) object 32kB dask.array<chunksize=(1980, 2), meta=np.ndarray>
Dimensions without coordinates: nbnd
Data variables:
    pr         (time, lat, lon) float32 438MB dask.array<chunksize=(600, 192, 288), meta=np.ndarray>
Attributes: (12/47)
    Conventions:            CF-1.7 CMIP-6.2
    activity_id:            CMIP
    branch_method:          standard
    branch_time_in_child:   674885.0
    branch_time_in_parent:  219000.0
    case_id:                972
    ...                     ...
    table_id:               Amon
    tracking_id:            hdl:21.14100/72897fd8-3516-431d-a1b1-c083130871a7...
    variable_id:            pr
    variant_info:           CMIP6 20th century experiments (1850-2014) with C...
    variant_label:          r11i1p1f1
    status:                 2019-10-25;created;by nhn2@columbia.edu

In [21]:
ds_historical['time'] = ds_historical.indexes['time'].to_datetimeindex()

/var/folders/cp/r3mkdy4d00l77xvrwh1q87nw0000gn/T/ipykernel_96056/1874523615.py:1: FutureWarning: In a future version of xarray to_datetimeindex will default to returning a 'us'-resolution DatetimeIndex instead of a 'ns'-resolution DatetimeIndex. This warning can be silenced by explicitly passing the `time_unit` keyword argument.
  ds_historical['time'] = ds_historical.indexes['time'].to_datetimeindex()
/var/folders/cp/r3mkdy4d00l77xvrwh1q87nw0000gn/T/ipykernel_96056/1874523615.py:1: RuntimeWarning: Converting a CFTimeIndex with dates from a non-standard calendar, 'noleap', to a pandas.DatetimeIndex, which uses dates from the standard calendar.  This may lead to subtle errors in operations that depend on the length of time between dates.
  ds_historical['time'] = ds_historical.indexes['time'].to_datetimeindex()


In [22]:
# Drop unnecessary coordinates and create 'year' coordinate
ds_historical = ds_historical.assign_coords(
    year = ds_historical.time.dt.year
)
ds_historical = ds_historical.drop_vars(['time', 'lat_bnds', 'lon_bnds', 'time_bnds'])
ds_historical

<xarray.Dataset> Size: 438MB
Dimensions:  (time: 1980, lat: 192, lon: 288)
Coordinates:
  * lat      (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon      (lon) float64 2kB 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.2 357.5 358.8
    year     (time) int64 16kB 1850 1850 1850 1850 1850 ... 2014 2014 2014 2014
Dimensions without coordinates: time
Data variables:
    pr       (time, lat, lon) float32 438MB dask.array<chunksize=(600, 192, 288), meta=np.ndarray>
Attributes: (12/47)
    Conventions:            CF-1.7 CMIP-6.2
    activity_id:            CMIP
    branch_method:          standard
    branch_time_in_child:   674885.0
    branch_time_in_parent:  219000.0
    case_id:                972
    ...                     ...
    table_id:               Amon
    tracking_id:            hdl:21.14100/72897fd8-3516-431d-a1b1-c083130871a7...
    variable_id:            pr
    variant_info:           CMIP6 20th century experiments (1850-2014) with C...
    variant_label:          r11i1p1f1
    status:                 2019-10-25;created;by nhn2@columbia.edu

In [17]:
# Return the precipitation for each lon and lat grouped by year
his_annual_precip = ds_historical.groupby('year').mean(dim='time')
his_annual_precip

<xarray.Dataset> Size: 37MB
Dimensions:  (year: 165, lat: 192, lon: 288)
Coordinates:
  * year     (year) int64 1kB 1850 1851 1852 1853 1854 ... 2011 2012 2013 2014
  * lat      (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon      (lon) float64 2kB 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.2 357.5 358.8
Data variables:
    pr       (year, lat, lon) float32 36MB dask.array<chunksize=(1, 192, 288), meta=np.ndarray>
Attributes: (12/47)
    Conventions:            CF-1.7 CMIP-6.2
    activity_id:            CMIP
    branch_method:          standard
    branch_time_in_child:   674885.0
    branch_time_in_parent:  219000.0
    case_id:                972
    ...                     ...
    table_id:               Amon
    tracking_id:            hdl:21.14100/72897fd8-3516-431d-a1b1-c083130871a7...
    variable_id:            pr
    variant_info:           CMIP6 20th century experiments (1850-2014) with C...
    variant_label:          r11i1p1f1
    status:                 2019-10-25;created;by nhn2@columbia.edu

In [18]:
#convert longitudes to -180 to 180 range
his_annual_precip = his_annual_precip.assign_coords(
    lon=(((his_annual_precip.lon + 180) % 360) - 180)
).sortby('lon')

**Compute the dataframe for each region's historical annual precipitation level**

In [37]:
northeast = his_annual_precip.sel(lat=slice(38,47), lon=slice(-80,-66))
south = his_annual_precip.sel(lat=slice(24,39), lon=slice(-106,-75))
midwest = his_annual_precip.sel(lat=slice(36,49), lon=slice(-104,-80))
west = his_annual_precip.sel(lat=slice(31,49), lon=slice(-125,-102))

In [ ]:
northeast_america_precip = northeast.groupby('year').mean(dim=['lat', 'lon'])
northeast_hist_df = northeast_america_precip.to_dataframe().reset_index()
northeast_hist_df

,year,pr
0,1850,0.000034
1,1851,0.000043
2,1852,0.000040
3,1853,0.000038
4,1854,0.000041
...,...,...
160,2010,0.000046
161,2011,0.000040
162,2012,0.000043
163,2013,0.000044


In [38]:
south_america_precip = south.groupby('year').mean(dim=['lat', 'lon'])
south_hist_df = south_america_precip.to_dataframe().reset_index()
south_hist_df

,year,pr
0,1850,0.000029
1,1851,0.000030
2,1852,0.000033
3,1853,0.000030
4,1854,0.000036
...,...,...
160,2010,0.000031
161,2011,0.000037
162,2012,0.000035
163,2013,0.000033


In [36]:
midwest_america_precip = midwest.groupby('year').mean(dim=['lat', 'lon'])
midwest_hist_df = midwest_america_precip.to_dataframe().reset_index()
midwest_hist_df

,year,pr
0,1850,0.000020
1,1851,0.000021
2,1852,0.000023
3,1853,0.000025
4,1854,0.000026
...,...,...
160,2010,0.000023
161,2011,0.000023
162,2012,0.000026
163,2013,0.000025


In [39]:
west_america_precip = west.groupby('year').mean(dim=['lat', 'lon'])
northwest_hist_df = west_america_precip.to_dataframe().reset_index()
northwest_hist_df

,year,pr
0,1850,0.000017
1,1851,0.000019
2,1852,0.000019
3,1853,0.000022
4,1854,0.000019
...,...,...
160,2010,0.000018
161,2011,0.000016
162,2012,0.000019
163,2013,0.000018


In [43]:
# Under a file named 'historical_data'
northeast_hist_df.to_csv('historical_data/northeast_historical_precipitation.csv', index=False)
south_hist_df.to_csv('historical_data/south_historical_precipitation.csv', index=False)
midwest_hist_df.to_csv('historical_data/midwest_historical_precipitation.csv', index=False)
northwest_hist_df.to_csv('historical_data/northwest_historical_precipitation.csv', index=False)

**Future precipitation level prediction under low emission scenario(SSP126) and high emission scenario(SSP585)**

In [40]:
future_low_emission = df[
   (df['activity_id'] == 'ScenarioMIP') &
   (df['institution_id'] == 'NCAR') &
   (df['source_id'] == 'CESM2') &
   (df['member_id'] == 'r11i1p1f1') &
   (df['experiment_id'] == 'ssp126') &
   (df['variable_id'].isin(['pr'])) &
   (df['table_id'].isin(['Amon'])) 
]
future_low_emission

,activity_id,institution_id,source_id,experiment_id,member_id,table_id,variable_id,grid_label,zstore,dcpp_init_year,version
444629,ScenarioMIP,NCAR,CESM2,ssp126,r11i1p1f1,Amon,pr,gn,gs://cmip6/CMIP6/ScenarioMIP/NCAR/CESM2/ssp126...,NaN,20200528


In [48]:
# Load future low emission scenario precipitation data
gcs = gcsfs.GCSFileSystem(token='anon')
zstore = future_low_emission.zstore.values[0]
ds_future_low = xr.open_zarr(gcs.get_mapper(zstore), consolidated=True)
ds_future_low['time'] = ds_future_low.indexes['time'].to_datetimeindex()
ds_future_low = ds_future_low.assign_coords(
    year = ds_future_low.time.dt.year
)
ds_future_low = ds_future_low.drop_vars(['time', 'lat_bnds', 'lon_bnds', 'time_bnds'])
ds_future_low = ds_future_low.groupby('year').mean(dim='time')
ds_future_low

/var/folders/cp/r3mkdy4d00l77xvrwh1q87nw0000gn/T/ipykernel_96056/1089699045.py:5: FutureWarning: In a future version of xarray to_datetimeindex will default to returning a 'us'-resolution DatetimeIndex instead of a 'ns'-resolution DatetimeIndex. This warning can be silenced by explicitly passing the `time_unit` keyword argument.
  ds_future_low['time'] = ds_future_low.indexes['time'].to_datetimeindex()
/var/folders/cp/r3mkdy4d00l77xvrwh1q87nw0000gn/T/ipykernel_96056/1089699045.py:5: RuntimeWarning: Converting a CFTimeIndex with dates from a non-standard calendar, 'noleap', to a pandas.DatetimeIndex, which uses dates from the standard calendar.  This may lead to subtle errors in operations that depend on the length of time between dates.
  ds_future_low['time'] = ds_future_low.indexes['time'].to_datetimeindex()


<xarray.Dataset> Size: 19MB
Dimensions:  (year: 86, lat: 192, lon: 288)
Coordinates:
  * year     (year) int64 688B 2015 2016 2017 2018 2019 ... 2097 2098 2099 2100
  * lat      (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon      (lon) float64 2kB 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.2 357.5 358.8
Data variables:
    pr       (year, lat, lon) float32 19MB dask.array<chunksize=(1, 192, 288), meta=np.ndarray>
Attributes: (12/46)
    Conventions:            CF-1.7 CMIP-6.2
    activity_id:            ScenarioMIP
    branch_method:          standard
    branch_time_in_child:   735110.0
    branch_time_in_parent:  735110.0
    case_id:                1728
    ...                     ...
    sub_experiment_id:      none
    table_id:               Amon
    tracking_id:            hdl:21.14100/b7d4e544-518d-4b38-84b5-4f48c432f3eb...
    variable_id:            pr
    variant_info:           CMIP6 SSP1-2.6 experiments (2015-2100) with CAM6,...
    variant_label:          r11i1p1f1

In [49]:
future_high_emission = df[
   (df['activity_id'] == 'ScenarioMIP') &
   (df['institution_id'] == 'NCAR') &
   (df['source_id'] == 'CESM2') &
   (df['member_id'] == 'r11i1p1f1') &
   (df['experiment_id'] == 'ssp585') &
   (df['variable_id'].isin(['pr'])) &
   (df['table_id'].isin(['Amon'])) 
]
future_high_emission

,activity_id,institution_id,source_id,experiment_id,member_id,table_id,variable_id,grid_label,zstore,dcpp_init_year,version
444743,ScenarioMIP,NCAR,CESM2,ssp585,r11i1p1f1,Amon,pr,gn,gs://cmip6/CMIP6/ScenarioMIP/NCAR/CESM2/ssp585...,NaN,20200528


In [50]:
# Load future high emission scenario precipitation data
gcs = gcsfs.GCSFileSystem(token='anon')    
zstore = future_high_emission.zstore.values[0]
ds_future_high = xr.open_zarr(gcs.get_mapper(zstore), consolidated=True)
ds_future_high['time'] = ds_future_high.indexes['time'].to_datetimeindex()
ds_future_high = ds_future_high.assign_coords(
    year = ds_future_high.time.dt.year
)
ds_future_high = ds_future_high.drop_vars(['time', 'lat_bnds', 'lon_bnds', 'time_bnds'])
ds_future_high = ds_future_high.groupby('year').mean(dim='time')
ds_future_high

/var/folders/cp/r3mkdy4d00l77xvrwh1q87nw0000gn/T/ipykernel_96056/529352229.py:5: FutureWarning: In a future version of xarray to_datetimeindex will default to returning a 'us'-resolution DatetimeIndex instead of a 'ns'-resolution DatetimeIndex. This warning can be silenced by explicitly passing the `time_unit` keyword argument.
  ds_future_high['time'] = ds_future_high.indexes['time'].to_datetimeindex()
/var/folders/cp/r3mkdy4d00l77xvrwh1q87nw0000gn/T/ipykernel_96056/529352229.py:5: RuntimeWarning: Converting a CFTimeIndex with dates from a non-standard calendar, 'noleap', to a pandas.DatetimeIndex, which uses dates from the standard calendar.  This may lead to subtle errors in operations that depend on the length of time between dates.
  ds_future_high['time'] = ds_future_high.indexes['time'].to_datetimeindex()


<xarray.Dataset> Size: 19MB
Dimensions:  (year: 86, lat: 192, lon: 288)
Coordinates:
  * year     (year) int64 688B 2015 2016 2017 2018 2019 ... 2097 2098 2099 2100
  * lat      (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon      (lon) float64 2kB 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.2 357.5 358.8
Data variables:
    pr       (year, lat, lon) float32 19MB dask.array<chunksize=(1, 192, 288), meta=np.ndarray>
Attributes: (12/46)
    Conventions:            CF-1.7 CMIP-6.2
    activity_id:            ScenarioMIP
    branch_method:          standard
    branch_time_in_child:   735110.0
    branch_time_in_parent:  735110.0
    case_id:                1734
    ...                     ...
    sub_experiment_id:      none
    table_id:               Amon
    tracking_id:            hdl:21.14100/7914233c-8b21-4cba-80fd-9398c24c50a4...
    variable_id:            pr
    variant_info:           CMIP6 SSP5-8.5 experiments (2015-2100) with CAM6,...
    variant_label:          r11i1p1f1